In [1]:
import torch

ckpt = torch.load("run2.pt", map_location="cpu", weights_only= False)

print("Keys inside checkpoint:")
print(ckpt.keys())

print("\nCONFIG:")
for k, v in ckpt["config"].items():
    print(k, ":", v)

state = ckpt["model_state_dict"]

print("\nModel weight keys:")
for k in list(state.keys())[:20]:
    print(k)

Keys inside checkpoint:
dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'config', 'best_val_loss', 'history'])

CONFIG:
DATA_DIRS : ['C:\\Users\\ACER NITRO\\FSL\\Training\\KEYPOINTS']
SEQUENCE_LENGTH : 48
BATCH_SIZE : 16
EPOCHS : 120
LR : 0.0003
WEIGHT_DECAY : 1e-05
DROPOUT : 0.4
PATIENCE : 20
VAL_SPLIT : 0.15
TEST_SPLIT : 0.15
NORM_MODE : none
AUG_ON : False
EXPORT_DIR : C:\Users\ACER NITRO\FSL\Training\Modified_LSTM\ModifiedLSTM_Exports\HybridLSTMGRU\20260222-004239
BEST_DIR : C:\Users\ACER NITRO\FSL\Training\Modified_LSTM\ModifiedLSTM_best
CLASSES : ['Color_Black', 'Color_Blue', 'Color_Green', 'Color_Orange', 'Color_Pink', 'Color_Red', 'Color_White', 'Color_Yellow', 'Family_Daughter', 'Family_Father', 'Family_Grandfather', 'Family_Grandmother', 'Family_Mother', 'Family_Son', 'Numbers_Five', 'Numbers_Four', 'Numbers_One', 'Numbers_Three', 'Numbers_Two', 'Relationship_Boy', 'Relationship_Girl', 'Survival_Correct', "Survival_Don't Understand", 'Survival_No', 'Survival_U

In [ ]:
import torch
import torch.nn as nn

class ModifiedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes,
                 dropout=0.35, use_layernorm=True):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.use_layernorm = use_layernorm

        self.lstm_layers = nn.ModuleList([
            nn.LSTM(input_size if i == 0 else hidden_size,
                    hidden_size, batch_first=True, dropout=0.0)
            for i in range(num_layers)
        ])

        if use_layernorm:
            self.layernorms = nn.ModuleList(
                [nn.LayerNorm(hidden_size) for _ in range(num_layers)]
            )

        self.act = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, reset_mask=None):
        out = x
        for i, lstm in enumerate(self.lstm_layers):
            out, _ = lstm(out)
            if self.use_layernorm:
                out = self.layernorms[i](out)
            out = self.act(out)
            out = self.drop(out)
            if reset_mask is not None:
                out = out * reset_mask.unsqueeze(-1)
        out = out.mean(dim=1)
        return self.fc(out)

ckpt = torch.load("run2.pt", map_location="cpu", weights_only=False)

config = ckpt["config"]

print("Loaded config:")
for k, v in config.items():
    if k in ["FEATURE_DIM", "HIDDEN_SIZE", "NUM_LAYERS", "DROPOUT", "SEQUENCE_LENGTH"]:
        print(f"{k}: {v}")

model = ModifiedLSTM(
    input_size=config["FEATURE_DIM"],
    hidden_size=config["HIDDEN_SIZE"],
    num_layers=config["NUM_LAYERS"],
    num_classes=len(config["CLASSES"]),
    dropout=config["DROPOUT"],
    use_layernorm=True
)

try:
    model.load_state_dict(ckpt["model_state_dict"])
    print("\n✅ SUCCESS — Architecture matches perfectly.")
except Exception as e:
    print("\n❌ MISMATCH DETECTED:")
    print(e)

Loaded config:
SEQUENCE_LENGTH: 48
DROPOUT: 0.4
FEATURE_DIM: 188
HIDDEN_SIZE: 384
NUM_LAYERS: 3

✅ SUCCESS — Architecture matches perfectly.


In [5]:
import onnx

model = onnx.load("model.onnx")
onnx.checker.check_model(model)

print("ONNX model is valid.")

ONNX model is valid.
